# Setup and data
Mount Drive, unpack the code-only archive from the README, and inspect the transfer plan.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import subprocess, sys, os
DATA = Path(os.getenv("V3_DATA", "/content/drive/MyDrive/audio-v3"))
REPO = Path(os.getenv("V3_REPO", "/content/v3-src"))
LOCAL = Path(os.getenv("V3_LOCAL", "/content/v3-local"))
REPO.mkdir(parents=True, exist_ok=True)
subprocess.run(["tar","-xf",str(DATA/"v3-code.tar"),"-C",str(REPO)], check=True)
sys.path.insert(0, str(REPO))
from colab.run_stage import run
EXECUTE = False  # Set True after reviewing this stage's data and compute estimate.


In [ ]:
if EXECUTE:
    subprocess.run([sys.executable,"-m","pip","install","-r",str(REPO/"pipeline_v3/requirements-colab.txt")],check=True)


The 25 November Drive already contains the publisher archives, models and development/selection.csv. Use the English repair below. Switch EXECUTE on for the action you intend to run. For a fresh Drive, first follow the original acquisition and inventory steps in the runbook.

In [ ]:
def module(name, *args):
    subprocess.run([sys.executable, "-m", "pipeline_v3." + name, *args, "--data", str(DATA)], cwd=REPO, check=True)
module("fetch", "--sources", "libri-dev", "--dry-run")
if EXECUTE:
    module("fetch", "--sources", "libri-dev", "--execute")
    module("english", "plan")
    module("generate", "--jobs", "english-pilot-jobs.csv", "--dry-run")

In [ ]:
GENERATE = False
if GENERATE:
    from google.colab import userdata
    for key in ("SPEECH_KEY", "SPEECH_REGION", "GEMINI_API_KEY"):
        os.environ[key] = userdata.get(key)
    subprocess.run([sys.executable, "-m", "pip", "install", "google-genai", "azure-cognitiveservices-speech"], check=True)
    module("generate", "--jobs", "english-pilot-jobs.csv", "--execute")

In [ ]:
PACK = False
if PACK:
    module("english", "assemble")
    module("cli", "catalog", "--selection", "selection-pilot.csv")
    module("cli", "pack")